# Step 3: Train RF-DETR on Swimming Pools

Trains RF-DETR Nano and Small via transfer learning on the manually-cleaned Roboflow COCO export, then compares against YOLO26 from Step 2.

Runtime: A100 GPU, ~30-40 min for both variants.

## Setup

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q 'rfdetr[train,loggers]>=1.4.0' supervision faster-coco-eval pandas matplotlib
import rfdetr
from importlib.metadata import version
print('rfdetr', version('rfdetr'))

## Load COCO dataset from Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile, pathlib

ZIP_PATH = '/content/drive/MyDrive/IE/CV/roboflow/Pools.coco.zip'
DATA_DIR = pathlib.Path('/content/dataset_coco')

DATA_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall(DATA_DIR)

for split in ['train', 'valid', 'test']:
    p = DATA_DIR / split / '_annotations.coco.json'
    print(f'{split}: {p.exists()}  {p}')

## Training configuration

| Parameter | Value |
|---|---|
| Backbone | DINOv2 ViT (pretrained, frozen for warm-up) |
| Optimizer | AdamW (RF-DETR default) |
| LR | 1e-4 head, 1e-5 backbone (RF-DETR defaults) |
| LR scheduler | Step decay (RF-DETR default) |
| Epochs | 100 |
| Batch size | 8, grad_accum=2 (effective 16, matches YOLO26) |
| Resolution | 672 (divisible by 14 and 56; closest valid value to the 640 of YOLO26) |
| Hardware | NVIDIA A100 40 GB |

In [ ]:
RESOLUTION = 672

TRAIN_CFG = dict(
    dataset_dir=str(DATA_DIR),
    epochs=100,
    batch_size=8,
    grad_accum_steps=2,
    seed=0,
)
for k, v in TRAIN_CFG.items(): print(f'{k:18s} = {v}')

## Train Nano and Small

In [ ]:
import time, gc, torch, pathlib
import torch.nn as nn
from rfdetr import RFDETRNano, RFDETRSmall

results = {}

def count_params(m):
    if isinstance(m, nn.Module):
        return sum(p.numel() for p in m.parameters())
    for attr in ('model', 'net', 'module'):
        sub = getattr(m, attr, None)
        if sub is not None:
            return count_params(sub)
    return 0

for name, cls in [('rfdetr_nano', RFDETRNano), ('rfdetr_small', RFDETRSmall)]:
    print(f'\n=== {name} ===')
    gc.collect(); torch.cuda.empty_cache()
    t0 = time.time()
    model = cls(resolution=RESOLUTION)
    out_dir = pathlib.Path(f'/content/runs/rfdetr/{name}')
    model.train(output_dir=str(out_dir), **TRAIN_CFG)
    best = out_dir / 'checkpoint_best_total.pth'
    if not best.exists():
        best = next(out_dir.glob('checkpoint_best*.pth'))
    results[name] = {
        'params': count_params(model),
        'train_time_s': time.time() - t0,
        'best_weights': str(best),
    }
    del model

## Validation and test metrics (mAP via supervision)

In [ ]:
import supervision as sv
from supervision.metrics import MeanAveragePrecision
from PIL import Image
import numpy as np

def iou_matrix(a, b):
    a = np.asarray(a, float); b = np.asarray(b, float)
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    bb = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    union = aa[:, None] + bb[None, :] - inter
    return np.where(union > 0, inter / union, 0)

def pr_at(preds, targets, conf=0.25, thr=0.5):
    TP = FP = FN = 0
    for d, t in zip(preds, targets):
        keep = d.confidence > conf
        db, dc = d.xyxy[keep], d.confidence[keep]
        gb = t.xyxy
        if len(gb) == 0: FP += len(db); continue
        if len(db) == 0: FN += len(gb); continue
        m = iou_matrix(db, gb); matched = set()
        for di in np.argsort(-dc):
            best_j, best_iou = -1, thr
            for gj in range(len(gb)):
                if gj in matched: continue
                if m[di, gj] >= best_iou: best_iou, best_j = m[di, gj], gj
            if best_j >= 0: TP += 1; matched.add(best_j)
            else: FP += 1
        FN += len(gb) - len(matched)
    P = TP / (TP + FP) if (TP + FP) else 0
    R = TP / (TP + FN) if (TP + FN) else 0
    return P, R, TP, FP, FN

def evaluate(model, split_dir):
    ds = sv.DetectionDataset.from_coco(
        images_directory_path=str(split_dir),
        annotations_path=str(split_dir / '_annotations.coco.json'),
    )
    preds, targets = [], []
    for path, _, gt in ds:
        img = Image.open(path).convert('RGB')
        det = model.predict(img, threshold=0.01)
        preds.append(det); targets.append(gt)
    mp = MeanAveragePrecision().update(preds, targets).compute()
    P, R, TP, FP, FN = pr_at(preds, targets)
    return {
        'mAP50':    float(mp.map50),
        'mAP50_95': float(mp.map50_95),
        'precision': float(P),
        'recall':    float(R),
        'TP': TP, 'FP': FP, 'FN': FN,
    }

for name, cls in [('rfdetr_nano', RFDETRNano), ('rfdetr_small', RFDETRSmall)]:
    print(f'\n--- {name} ---')
    model = cls(resolution=RESOLUTION, pretrain_weights=results[name]['best_weights'])
    model.optimize_for_inference()
    results[name]['val']  = evaluate(model, DATA_DIR / 'valid')
    results[name]['test'] = evaluate(model, DATA_DIR / 'test')
    print('  val ', results[name]['val'])
    print('  test', results[name]['test'])
    del model
    gc.collect(); torch.cuda.empty_cache()

## Comparison table (RF-DETR vs YOLO26)

In [ ]:
import pandas as pd

rows = []
for name, r in results.items():
    rows.append({
        'model': name,
        'mAP@50':    r['val']['mAP50'],
        'mAP@50-95': r['val']['mAP50_95'],
        'Precision': r['val']['precision'],
        'Recall':    r['val']['recall'],
        'Params (M)': r['params'] / 1e6,
        'Train time (s)': r['train_time_s'],
    })
df_rf = pd.DataFrame(rows).set_index('model').round(4)

YOLO_CSV = '/content/drive/MyDrive/IE/CV/results/yolo26_hbb/comparison.csv'
try:
    df_yolo = pd.read_csv(YOLO_CSV, index_col=0)
    df_combined = pd.concat([df_yolo, df_rf])
    print('Combined comparison (YOLO26 + RF-DETR):')
    print(df_combined.to_string())
    df_combined.to_csv('/content/rfdetr_combined_comparison.csv')
except FileNotFoundError:
    print(f'YOLO26 CSV not found at {YOLO_CSV} -- run Step 2 first to get the comparison')
    print(df_rf.to_string())
df_rf

## Failure analysis (best variant, test set)

In [ ]:
import cv2, matplotlib.pyplot as plt, pathlib

BEST = max(results, key=lambda n: results[n]['val']['mAP50_95'])
print(f'Best RF-DETR variant: {BEST}')

cls = {'rfdetr_nano': RFDETRNano, 'rfdetr_small': RFDETRSmall}[BEST]
model = cls(resolution=RESOLUTION, pretrain_weights=results[BEST]['best_weights'])
model.optimize_for_inference()

test_dir = DATA_DIR / 'test'
test_ds = sv.DetectionDataset.from_coco(
    images_directory_path=str(test_dir),
    annotations_path=str(test_dir / '_annotations.coco.json'),
)

CONF, IOU_TH = 0.25, 0.5
fps, fns = [], []  # (img_path, box, score_or_area)

for path, _, gt in test_ds:
    img = Image.open(path).convert('RGB')
    det = model.predict(img, threshold=CONF)
    db, dc = det.xyxy, det.confidence
    gb = gt.xyxy
    matched_gt, matched_pr = set(), set()
    if len(db) and len(gb):
        m = iou_matrix(db, gb)
        for di in np.argsort(-dc):
            best_j, best_iou = -1, IOU_TH
            for gj in range(len(gb)):
                if gj in matched_gt: continue
                if m[di, gj] >= best_iou: best_iou, best_j = m[di, gj], gj
            if best_j >= 0: matched_gt.add(best_j); matched_pr.add(int(di))
    for i in range(len(db)):
        if i not in matched_pr: fps.append((path, db[i].tolist(), float(dc[i])))
    for j in range(len(gb)):
        if j not in matched_gt:
            area = (gb[j, 2] - gb[j, 0]) * (gb[j, 3] - gb[j, 1])
            fns.append((path, gb[j].tolist(), float(area)))

fps.sort(key=lambda x: -x[2])
fns.sort(key=lambda x: -x[2])
print(f'False positives: {len(fps)}   False negatives: {len(fns)}')

def show(items, title, n=5):
    fig, axes = plt.subplots(1, n, figsize=(4*n, 4))
    for ax, (p, box, _) in zip(axes, items[:n]):
        img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
        ax.imshow(img); ax.set_title(pathlib.Path(p).name, fontsize=8); ax.axis('off')
    fig.suptitle(title); plt.tight_layout(); plt.show()

show(fps, f'{BEST}: top-5 false positives')
show(fns, f'{BEST}: top-5 false negatives')

## Save results to Drive

In [ ]:
import shutil

OUT = pathlib.Path('/content/drive/MyDrive/IE/CV/results/rfdetr')
OUT.mkdir(parents=True, exist_ok=True)

df_rf.to_csv(OUT / 'comparison.csv')
for name, r in results.items():
    src = pathlib.Path(r['best_weights'])
    if src.exists():
        shutil.copy(src, OUT / f'{name}_best.pth')
print(f'Saved to {OUT}')